In [1]:
import torch
import torch.nn as nn

In [50]:
EMBEDDING_DIN = 768
CONTEXT_LENGTH = 1024
BATCH = 4
inputs = torch.rand(BATCH,CONTEXT_LENGTH,EMBEDDING_DIN)
inputs.shape,inputs[0].shape,inputs[0,0,:].shape

(torch.Size([4, 1024, 768]), torch.Size([1024, 768]), torch.Size([768]))

In [3]:
torch.transpose?

Docstring:
transpose(input, dim0, dim1) -> Tensor

Returns a tensor that is a transposed version of :attr:`input`.
The given dimensions :attr:`dim0` and :attr:`dim1` are swapped.

If :attr:`input` is a strided tensor then the resulting :attr:`out`
tensor shares its underlying storage with the :attr:`input` tensor, so
changing the content of one would change the content of the other.

If :attr:`input` is a :ref:`sparse tensor <sparse-docs>` then the
resulting :attr:`out` tensor *does not* share the underlying storage
with the :attr:`input` tensor.

If :attr:`input` is a :ref:`sparse tensor <sparse-docs>` with compressed
layout (SparseCSR, SparseBSR, SparseCSC or SparseBSC) the arguments
:attr:`dim0` and :attr:`dim1` must be both batch dimensions, or must
both be sparse dimensions. The batch dimensions of a sparse tensor are the
dimensions preceding the sparse dimensions.

.. note::
    Transpositions which interchange the sparse dimensions of a `SparseCSR`
    or `SparseCSC` layout tens

# 1 Simple Attention

In [4]:
class NaiveAttention(torch.nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,x):
        values = (x@x.transpose(1,2))@x
        print(values.shape)

In [5]:
naive = NaiveAttention()
naive.eval()
with torch.no_grad():
    naive(inputs)

torch.Size([4, 100, 64])


# 2 Scaled attention with K,Q,V projections & single Head

In [6]:
class attentionKQV(torch.nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        # input shape -> [ B , T , D_in]
        self.d_out = d_out
        self.Q = nn.Parameter(torch.rand(d_in,d_out))
        self.K = nn.Parameter(torch.rand(d_in,d_out))
        self.V = nn.Parameter(torch.rand(d_in,d_out))
    
    def forward(self,x):
        q_proj = x@self.Q  # [ B , T , D_out]
        k_proj = x@self.K  # [ B , T , D_out]
        v_proj = x@self.V  # [ B , T , D_out]
        attention = torch.softmax(q_proj @ k_proj.transpose(1,2) / q_proj.shape[-1]**0.5,dim=2)
        print(attention.shape,"attention shape")
        print(attention.sum())
        z_vector = attention@v_proj
        print(z_vector.shape , "context shape")
        return z_vector        

In [7]:
torch.manual_seed(123)
kqv = attentionKQV(64,16)
z = kqv(inputs)
print("\n")

torch.Size([4, 100, 100]) attention shape
tensor(400., grad_fn=<SumBackward0>)
torch.Size([4, 100, 16]) context shape




# 2.1 Scaled attention with K,Q,V projections & single Head , using Linear Layer

In [8]:
class attentionKQV_2(torch.nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        # input shape -> [ B , T , D_in]
        self.d_out = d_out
        self.Q = nn.Linear(d_in,d_out,bias=False)
        self.K = nn.Linear(d_in,d_out,bias=False)
        self.V = nn.Linear(d_in,d_out,bias=False)
        print(self.Q.weight.shape)
    
    def forward(self,x):
        q_proj = self.Q(x)  # [ B , T , D_out]  ( B , T , D_in)  -> [ B , T , D_out]
        k_proj = self.K(x)
        v_proj = self.V(x)
        print(q_proj.shape)
        attention = torch.softmax(q_proj @ k_proj.transpose(1,2) / q_proj.shape[-1]**0.,dim=2)
        print(attention.shape,"attention shape")
        print(attention.sum())
        z_vector = attention@v_proj
        print(z_vector.shape , "context shape")
        return z_vector        
         

In [ ]:
torch.manual_seed(123)
kqv_2 = attentionKQV_2(EMBEDDING_DIN,16)
z = kqv_2(inputs)
print("\n")

torch.Size([16, 64])
torch.Size([4, 100, 16])
torch.Size([4, 100, 100]) attention shape
tensor(400., grad_fn=<SumBackward0>)
torch.Size([4, 100, 16]) context shape




# 3 Causal attention with single head

In [10]:
class CausalAttention(torch.nn.Module):
    def __init__(self,d_in,d_out,drop_rate):
        super().__init__()
        self.Q = nn.Linear(d_in,d_out,bias=False)
        self.K = nn.Linear(d_in,d_out,bias=False)
        self.V = nn.Linear(d_in,d_out,bias=False)
        self.dropout = nn.Dropout(drop_rate)
        self.register_buffer('mask', torch.triu(torch.ones(CONTEXT_LENGTH, CONTEXT_LENGTH), diagonal=1))
        print(self.Q.weight.shape)
    
    def forward(self,x):
        # expected x -> [B,T,d_in]
        #          K,Q,V -> [B,d_in,d_out]
        q_proj = self.Q(x) #Expexted [B,T,d_out]
        k_proj = self.K(x)
        v_proj = self.V(x)
        attn_scores = q_proj@k_proj.transpose(-2,-1)
        attn_scores.masked_fill_(self.mask.bool()[:CONTEXT_LENGTH, :CONTEXT_LENGTH], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attention_weights = torch.softmax(attn_scores/ q_proj.shape[-1]**0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)
        return attention_weights@v_proj
        

In [ ]:
ca = CausalAttention(EMBEDDING_DIN,16,0.3)
ca(inputs).shape

torch.Size([16, 64])


torch.Size([4, 100, 16])

# 4 MHA

In [29]:
class MHA(nn.Module):
    def __init__(self,d_in,d_out,drop_rate,num_heads):
        super().__init__()
        assert(d_in%num_heads==0)
        self.Q = nn.Linear(d_in,d_out,bias = False)
        self.K = nn.Linear(d_in,d_out,bias = False)
        self.V = nn.Linear(d_in,d_out,bias = False)
        self.dropout = nn.Dropout(drop_rate)
        self.num_heads = num_heads
        self.head_dim = d_in//num_heads
        self.cl = CONTEXT_LENGTH
        self.register_buffer('mask', torch.triu(torch.ones(CONTEXT_LENGTH, CONTEXT_LENGTH), diagonal=1)) # New
    
    def forward(self,x):
        # expected x -> [B,T,d_in]
        #          K,Q,V -> [B,d_in,d_out]
        q_proj = self.Q(x) #Expexted [B,T,d_out]
        k_proj = self.K(x)
        v_proj = self.V(x)
        q_proj = q_proj.view(q_proj.shape[0],q_proj.shape[1],self.num_heads,q_proj.shape[-1]//self.num_heads)
        k_proj = k_proj.view(k_proj.shape[0],k_proj.shape[1],self.num_heads,k_proj.shape[-1]//self.num_heads)
        v_proj = v_proj.view(v_proj.shape[0],v_proj.shape[1],self.num_heads,v_proj.shape[-1]//self.num_heads)
        q_proj = q_proj.transpose(1,2) # [ B,H,T,D/H]
        k_proj = k_proj.transpose(1,2)
        v_proj = v_proj.transpose(1,2)
        
        attention_score = q_proj@k_proj.transpose(-2,-1) # [B,H,T,T]
        attention_score.masked_fill_(self.mask.bool()[:v_proj.shape[-2],:v_proj.shape[-2]],-torch.inf)
        attention_weights = torch.softmax(attention_score / k_proj.shape[-1]**0.5,dim=-1)
        attention_weights = self.dropout(attention_weights) # [ B,H,T,T]
        context =  attention_weights@v_proj  # [ B,H,T,D/H ]
        context = context.transpose(1,2)
        return context.contiguous().view(context.shape[0],context.shape[1],-1)

In [52]:
mha = MHA(EMBEDDING_DIN,768,0.3,12)
mha(inputs).shape

torch.Size([4, 1024, 768])

In [53]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [54]:
count_parameters(mha)

1769472

i.e. 17 Lakh parameters 

Scratch pad

In [15]:
a = torch.randn(1, 2, 3, 4)

In [16]:
b = a.view(1,3,2,4)

In [19]:
c = b.transpose(1,2)

In [20]:
b.shape,c.shape

(torch.Size([1, 3, 2, 4]), torch.Size([1, 2, 3, 4]))

In [23]:
torch.all(a==c)

tensor(False)

In [42]:
vals = torch.tensor([i for i in range(256)])
vals.shape

torch.Size([256])

In [43]:
vals = vals.reshape(4,4,4,4)
vals[0,1,2,3],vals.shape

(tensor(27), torch.Size([4, 4, 4, 4]))

In [44]:
vals_view = vals.view(1,1,1,256)
vals_view[0,0,0,1*4**2+2*4+3] , vals.shape

(tensor(27), torch.Size([4, 4, 4, 4]))

In [47]:
vals_view2 = vals.transpose(1,2)
vals_view2[0,1,2,3] , vals_view2[0,2,1,3]

(tensor(39), tensor(27))